In [4]:
import os
import time
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
import haversine as hs
from tqdm import tqdm, trange
from dotenv import load_dotenv

# Testing Queries for Seiscomp6

In previous versions of the revision routine, there are a fixed set of queries that are used to test the connection to the database and to extract the info needed to run the revision. Here, we test the same queries for Seiscomp6, and make sure they work as expected. We will also test the connection to the database, and make sure that we can ping the server. In addition, by using DBeaver I try to extend the queries to extract more info that may be useful for the revision, mainly for those events with 6 to 8 phases.

Let's creating a single function that queries some info from the seiscomp database.

In [5]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

def connect_to_db(
        query: str,
        start_time: dt.datetime = None,
        end_time: dt.datetime = None,
        **kwargs):

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        full_query = f"{query} '{start_time_str}' and '{end_time_str}' ORDER BY Origin.time_value ASC;"  # Filter and order by time_value
    else:
        full_query = query

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        db_connection = pymysql.connect(
            host=os.getenv('SERVER_SC3_HOST'),
            user=os.getenv('SERVER_SC3_USERNAME'),
            password=os.getenv('SERVER_SC3_PASSWORD'),
            database=os.getenv('SERVER_SC3_DATABASE')
        )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                df = pd.read_sql_query(full_query, db_connection, **kwargs)
                pbar.update(1)
        finally:
            db_connection.close()

    return df

# Unboxing current queries

Let's start by unboxing the current queries used in the revision routine, and test them one by one.

## Normal queries

The SQL query used is:

```sql
Select Origin.time_value, POEv.publicID, Origin.depth_value, Magnitude.magnitude_value, Origin.quality_standardError, Origin.depth_uncertainty, Origin.latitude_uncertainty, Origin.longitude_uncertainty, Origin.quality_associatedPhaseCount, Origin.creationInfo_author, Event.type, Origin.creationInfo_agencyID, EventDescription.text, Origin.latitude_value, Origin.longitude_value, Magnitude.type, Origin.methodID, Origin.earthModelID from Event AS EvMF left join PublicObject AS POEv ON EvMF._oid = POEv._oid left join PublicObject as POOri ON EvMF.preferredOriginID=POOri.publicID left join Origin ON POOri._oid=Origin._oid left join PublicObject as POMag on EvMF.preferredMagnitudeID=POMag.publicID left join Magnitude ON Magnitude._oid = POMag._oid left join Event ON Event._oid= POEv._oid left join EventDescription ON EvMF._oid = EventDescription._parent_oid where Origin.time_value between
```

where:

- `EvMF` is the Event table, which contains the main information about the event, such as the origin time, depth, magnitude, etc.
- `POEv` is the PublicObject table, which contains the public ID of the event, which is used to link the Event table with the Origin and Magnitude tables.
- `POOri` is the PublicObject table, which contains the public ID of the origin, which is used to link the Origin table with the Event table.
- `Origin` is the Origin table, which contains the information about the origin of the event, such as the time, depth, latitude, longitude, etc.
- `POMag` is the PublicObject table, which contains the public ID of the magnitude, which is used to link the Magnitude table with the Event table.
- `Magnitude` is the Magnitude table, which contains the information about the magnitude of the event, such as the magnitude value, type, etc.
- `EventDescription` is the EventDescription table, which contains the description of the event, such as the text description, etc.

In this SQL query, we are selecting the following fields:
- `Origin.time_value`: the origin time of the event
- `POEv.publicID`: the public ID of the event
- `Origin.depth_value`: the depth of the event
- `Magnitude.magnitude_value`: the magnitude of the event
- `Origin.quality_standardError`: the standard error of the origin
- `Origin.depth_uncertainty`: the uncertainty of the depth
- `Origin.latitude_uncertainty`: the uncertainty of the latitude
- `Origin.longitude_uncertainty`: the uncertainty of the longitude
- `Origin.quality_associatedPhaseCount`: the number of associated phases
- `Origin.creationInfo_author`: the author of the origin
- `Event.type`: the type of the event
- `Origin.creationInfo_agencyID`: the agency ID of the origin
- `EventDescription.text`: the text description of the event
- `Origin.latitude_value`: the latitude of the event
- `Origin.longitude_value`: the longitude of the event
- `Magnitude.type`: the type of the magnitude
- `Origin.methodID`: the method ID of the origin
- `Origin.earthModelID`: the earth model ID of the origin

and the condition is that the origin time of the event is between a certain time range, which is specified in the last part of the query.

Let's test this query directly in this notebook, by using the function from the last cell. We will see as a simple example the query for a specific time range, to see if we can retrieve all the origins that have been created between a time range.

In [6]:
query_example = "SELECT * FROM Origin WHERE Origin.time_value BETWEEN"
origin_df = connect_to_db(query_example, start_time=dt.datetime(2026, 5, 1), end_time=dt.datetime.now(dt.UTC))
origin_df

,_oid,_parent_oid,_last_modified,time_value,time_value_ms,time_uncertainty,time_lowerUncertainty,time_upperUncertainty,time_confidenceLevel,time_pdf_variable_content,...,creationInfo_agencyID,creationInfo_agencyURI,creationInfo_author,creationInfo_authorURI,creationInfo_creationTime,creationInfo_creationTime_ms,creationInfo_modificationTime,creationInfo_modificationTime_ms,creationInfo_version,creationInfo_used


To check, as another example, the info from all preferred origins between two time ranges, we can use the same query stated earlier:

In [7]:
initial_time = dt.datetime(2023, 5, 3, 0, 0, 0)
final_time = dt.datetime(2026, 3, 17, 0, 0, 0)

In [8]:
query2 = "Select Origin.time_value, POEv.publicID, Origin.depth_value, Magnitude.magnitude_value, Origin.quality_standardError, Origin.depth_uncertainty, Origin.latitude_uncertainty, Origin.longitude_uncertainty, Origin.quality_associatedPhaseCount, Origin.creationInfo_author, Event.type, Origin.creationInfo_agencyID, EventDescription.text, Origin.latitude_value, Origin.longitude_value, Magnitude.type, Origin.methodID, Origin.earthModelID from Event AS EvMF left join PublicObject AS POEv ON EvMF._oid = POEv._oid left join PublicObject as POOri ON EvMF.preferredOriginID=POOri.publicID left join Origin ON POOri._oid=Origin._oid left join PublicObject as POMag on EvMF.preferredMagnitudeID=POMag.publicID left join Magnitude ON Magnitude._oid = POMag._oid left join Event ON Event._oid= POEv._oid left join EventDescription ON EvMF._oid = EventDescription._parent_oid where Origin.time_value between"
event_df2 = connect_to_db(query2, start_time=initial_time, end_time=final_time)
event_df2

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,type,creationInfo_agencyID,text,latitude_value,longitude_value,type,methodID,earthModelID
0,2023-05-03 00:16:51,SGC2023ipzdxf,10.000000,NaN,10.420535,NaN,NaN,NaN,NaN,eguzman@proc3,not locatable,SGC,"CalarcÃ¡ - QuindÃ­o, Colombia",4.484200,-75.636700,None,,
1,2023-05-03 00:22:16,SGC2023ipzion,-4.941406,0.885335,0.445758,5.790340,2.080092,5.282553,8.0,eguzman@proc3,not locatable,SGC,"Colombia-Ecuador, Region Fronteriza",0.781972,-77.914268,MLr,NonLinLoc,Poveda_et_al_2018
2,2023-05-03 01:17:28,SGC2023iqbedm,135.859375,1.911117,0.959775,9.021321,6.419839,7.434003,19.0,eguzman@proc3,earthquake,SGC,"Zapatoca - Santander, Colombia",6.766267,-73.218474,MLr_3,NonLinLoc,Poveda_et_al_2018
3,2023-05-03 01:24:17,SGC2023iqbkaq,22.539062,0.880860,0.356193,7.230936,5.458100,3.798461,12.0,eguzman@proc3,earthquake,SGC,"PurificaciÃ³n - Tolima, Colombia",3.844233,-74.806306,MLr_2,NonLinLoc,Poveda_et_al_2018
4,2023-05-03 01:28:27,SGC2023iqbnpk,10.000000,NaN,3.682291,NaN,NaN,NaN,NaN,eguzman@proc3,not locatable,SGC,Mar Caribe,8.644500,-77.360400,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209266,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,hmoreno@proc2,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018
209267,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,hmoreno@proc2,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018
209268,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,hmoreno@proc2,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,
209269,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,hmoreno@proc2,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,


However, it is more convenient to read the query from a .sql file, to avoid having a very long query in the notebook. We can read the query from a .sql file, and then use it in the function. Let's do it for the previous query, but adding the following columns:

1. The used stations for the localization.
2. The total stations that have associated phases.
3. The used phases for the localization.
4. The Comments section used to define a DESTACADO event.

In [9]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

In [10]:
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)
event_df3

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2023-05-03 00:16:51,SGC2023ipzdxf,10.000000,NaN,10.420535,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"CalarcÃ¡ - QuindÃ­o, Colombia",4.484200,-75.636700,None,,,None
1,2023-05-03 00:22:16,SGC2023ipzion,-4.941406,0.885335,0.445758,5.790340,2.080092,5.282553,8.0,8.0,...,NaN,not locatable,SGC,"Colombia-Ecuador, Region Fronteriza",0.781972,-77.914268,MLr,NonLinLoc,Poveda_et_al_2018,None
2,2023-05-03 01:17:28,SGC2023iqbedm,135.859375,1.911117,0.959775,9.021321,6.419839,7.434003,19.0,19.0,...,NaN,earthquake,SGC,"Zapatoca - Santander, Colombia",6.766267,-73.218474,MLr_3,NonLinLoc,Poveda_et_al_2018,None
3,2023-05-03 01:24:17,SGC2023iqbkaq,22.539062,0.880860,0.356193,7.230936,5.458100,3.798461,12.0,12.0,...,NaN,earthquake,SGC,"PurificaciÃ³n - Tolima, Colombia",3.844233,-74.806306,MLr_2,NonLinLoc,Poveda_et_al_2018,None
4,2023-05-03 01:28:27,SGC2023iqbnpk,10.000000,NaN,3.682291,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,Mar Caribe,8.644500,-77.360400,None,,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209266,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,27.0,...,NaN,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209267,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,22.0,...,NaN,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209268,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
209269,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None


By using this new SQL query, it is not required to use the previous two querys to select DESTACADO and normal events, since all the info is contained in the same query.

## Making SQL queries from seiscomp3 and seiscomp6 simultaneously

Since 2026-03-17 00:00:00 UTC, the new implementation of seiscomp6 is running, and all the events that are created from that moment on are stored in the new database. However, there are still many events that were created before that moment, which are stored in the old database. Therefore, to have a complete revision, it is necessary to query both databases simultaneously, and then merge the results. This can be done by using the same function to query both databases, and then concatenating the results into a single dataframe. Let's do it for the previous query, but for both databases.

In [11]:
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

# Cutover date: SC3 → SC6
SC6_CUTOVER = dt.datetime(2026, 3, 17, 0, 0, 0)

def _build_connection(prefix: str):
    """Create a pymysql connection using .env credentials for a given prefix."""
    return pymysql.connect(
        host=os.getenv(f'SERVER_{prefix}_HOST'),
        port=int(os.getenv(f'SERVER_{prefix}_PORT', 3306)),
        user=os.getenv(f'SERVER_{prefix}_USERNAME'),
        password=os.getenv(f'SERVER_{prefix}_PASSWORD'),
        database=os.getenv(f'SERVER_{prefix}_DATABASE'),
    )


def _query_db(
    prefix: str,
    query: str,
    start_time: dt.datetime,
    end_time: dt.datetime,
    desc: str,
    **kwargs,
) -> pd.DataFrame:
    """
    Execute a time-bounded SQL query against a single database.

    Parameters
    ----------
    prefix : str
        Credential prefix — 'SC3' or 'SC6'.
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time, end_time : datetime
        Time bounds for the query.
    desc : str
        Label shown in the tqdm progress bar.
    """
    start_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
    end_str   = end_time.strftime("%Y-%m-%d %H:%M:%S")
    full_query = (
        f"{query} '{start_str}' AND '{end_str}' "
        f"ORDER BY Origin.time_value ASC;"
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        conn = _build_connection(prefix)
        try:
            with tqdm(
                total=1,
                desc=desc,
                unit="query",
                leave=False,
                bar_format="{desc}",
            ) as pbar:
                df = pd.read_sql_query(full_query, conn, **kwargs)
                pbar.update(1)
        finally:
            conn.close()

    return df

def _to_naive_utc(t: dt.datetime) -> dt.datetime:
    """
    Normalize a datetime to naive UTC.
    - Timezone-aware → convert to UTC, strip tzinfo
    - Naive → assumed UTC already, returned as-is
    """
    if t.tzinfo is not None:
        return t.astimezone(dt.timezone.utc).replace(tzinfo=None)
    return t

def connect_to_db(
    query: str,
    start_time: dt.datetime = None,
    end_time: dt.datetime = None,
    **kwargs,
) -> pd.DataFrame:
    """
    Query SC3 and/or SC6 databases depending on the requested time range.

    Decision logic:
        - end_time   <= SC6_CUTOVER  → SC3 only
        - start_time >= SC6_CUTOVER  → SC6 only
        - start_time <  SC6_CUTOVER  < end_time → both (with overlap warning)

    Parameters
    ----------
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time : datetime, optional
        Start of the time range (UTC). If None, runs the query as-is.
    end_time : datetime, optional
        End of the time range (UTC). If None, runs the query as-is.

    Returns
    -------
    pd.DataFrame
        Query results, merged and sorted by time_value when both DBs are hit.
    """
    # --- No time range: run query as-is against SC3 (legacy default) ----------
    if start_time is None or end_time is None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            conn = _build_connection("SC3")
            try:
                with tqdm(
                    total=1,
                    desc="Querying database...",
                    unit="query",
                    leave=False,
                    bar_format="{desc}",
                ) as pbar:
                    df = pd.read_sql_query(query, conn, **kwargs)
                    pbar.update(1)
            finally:
                conn.close()
        return df

    # --- Normalize to naive UTC before any comparison ------------------------
    start_time = _to_naive_utc(start_time)
    end_time   = _to_naive_utc(end_time)

    # --- Determine which databases are needed --------------------------------
    only_sc3 = end_time   <= SC6_CUTOVER
    only_sc6 = start_time >= SC6_CUTOVER
    both     = not only_sc3 and not only_sc6       # straddles the cutover

    if both:
        warnings.warn(
            f"\n[DATABASE WARNING] The requested time range "
            f"({start_time:%Y-%m-%d %H:%M:%S} → {end_time:%Y-%m-%d %H:%M:%S}) "
            f"spans the seiscomp3-seiscomp6 database cutover ({SC6_CUTOVER:%Y-%m-%d %H:%M:%S} UTC). "
            f"Both databases will be queried and results merged.\n",
            UserWarning,
            stacklevel=2,
        )

    # --- SC3 only ------------------------------------------------------------
    if only_sc3:
        return _query_db(
            prefix="SC3",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC3 database...",
            **kwargs,
        )

    # --- SC6 only ------------------------------------------------------------
    if only_sc6:
        return _query_db(
            prefix="SC6",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC6 database...",
            **kwargs,
        )

    # --- Both databases (straddles cutover) ----------------------------------
    # SC3: [start_time, SC6_CUTOVER)
    # SC6: [SC6_CUTOVER, end_time]
    df_sc3 = _query_db(
        prefix="SC3",
        query=query,
        start_time=start_time,
        end_time=SC6_CUTOVER,
        desc="Querying SC3 database (1/2)...",
        **kwargs,
    )
    df_sc6 = _query_db(
        prefix="SC6",
        query=query,
        start_time=SC6_CUTOVER,
        end_time=end_time,
        desc="Querying SC6 database (2/2)...",
        **kwargs,
    )

    # Merge and re-sort by time_value
    df = (
        pd.concat([df_sc3, df_sc6], ignore_index=True)
        .sort_values("time_value")
        .reset_index(drop=True)
    )

    return df

In [12]:
# Test query for both cases
# Only SC3
initial_time_2 = dt.datetime(2026, 3, 16, 0, 0, 0)  # One day before cutover
final_time_2   = dt.datetime(2026, 3, 17, 0, 0, 0)  # One day after cutover
event_df_sc3 = connect_to_db(clean_sql, start_time=initial_time_2, end_time=final_time_2)
event_df_sc3

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2026-03-16 00:05:40,SGC2026ffvyal,5.000000,1.124101,0.010000,NaN,NaN,NaN,5.0,5.0,...,NaN,not locatable,SGC,"MutatÃ¡ - Antioquia, Colombia",7.199500,-76.386167,MLr_1,Hypo71,RSNC,None
1,2026-03-16 00:12:09,SGC2026ffwdqf,55.820312,1.307294,0.483821,5.940549,3.685058,5.293564,14.0,14.0,...,NaN,earthquake,SGC,"Hispania - Antioquia, Colombia",5.768267,-75.933496,MLr_1,NonLinLoc,Poveda_et_al_2018,None
2,2026-03-16 00:12:21,SGC2026ffwdtx,29.000000,2.412485,0.900000,3.000000,1.343503,1.343503,44.0,44.0,...,NaN,earthquake,SGC,OcÃ©ano PacÃ­fico,6.161000,-77.876333,MLr_1,Hypo71,RSNC,None
3,2026-03-16 00:37:31,SGC2026ffwzlw,10.000000,0.046732,8.279938,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
4,2026-03-16 00:46:40,SGC2026ffxhio,29.000000,4.181091,0.959388,0.000000,3.982650,4.372885,25.0,23.0,...,13.0,earthquake,SGC,OcÃ©ano PacÃ­fico,4.034341,-82.546722,mb,LOCSAT,iasp91,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,27.0,...,NaN,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018,None
231,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,22.0,...,NaN,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018,None
232,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
233,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None


In [13]:
# Only SC6
initial_time_3 = dt.datetime(2026, 3, 17, 0, 0, 0)  # Cutover time
final_time_3   = dt.datetime(2026, 3, 18, 0, 0, 0)  # One day after cutover
event_df_sc6 = connect_to_db(clean_sql, start_time=initial_time_3, end_time=final_time_3)
event_df_sc6

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2026-03-17 00:00:00,USGS_nc75329017,2.00,3.300000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,outside of network interest,USGS,"The Geysers, CA",38.829000,-122.823000,M,smi:local/manual,,None
1,2026-03-17 00:02:41,SGC2026fhroii,10.00,1.766955,17.281041,NaN,NaN,NaN,NaN,NaN,...,NaN,not existing,SGC,"Riosucio - Caldas, Colombia",5.421100,-75.710400,MLr_2,,,None
2,2026-03-17 00:16:44,SGC2026fhsalj,9.99,1.339800,0.260000,2.7,0.777817,0.777817,16.0,16.0,...,NaN,earthquake,SGC,"Beltrán - Cundinamarca, Colombia",4.834333,-74.748333,MLr_2,Hypo71,RSNC,None
3,2026-03-17 00:50:53,SGC2026fhtdwg,10.00,0.015015,6.615750,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
4,2026-03-17 00:51:44,SGC2026fhtepm,155.89,1.606607,0.190000,3.6,3.818377,3.818377,6.0,6.0,...,NaN,not locatable,SGC,"Los Santos - Santander, Colombia",6.826167,-73.084500,MLr_3,Hypo71,RSNC,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,2026-03-17 22:17:03,SGC2026fjjuen,10.00,-0.025865,4.241400,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
224,2026-03-17 22:23:34,SGC2026fjjzur,80.00,1.558878,0.190000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Convención - Norte de Santander, Colombia",8.781667,-73.233000,MLr_3,Hypo71,RSNC,None
225,2026-03-17 22:42:15,SGC2026fjkpxl,5.00,0.341099,0.100000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Puerto Gaitán - Meta, Colombia",3.882167,-71.381000,MLr_PtoGtn,Hypo71,RSNC,None
226,2026-03-17 23:01:23,SGC2026fjlgjx,5.00,1.129640,0.060000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Tibasosa - Boyacá, Colombia",5.748000,-73.009833,MLr_3,Hypo71,RSNC,None


In [14]:
# Both SC3 and SC6 (straddles cutover)
initial_time_4 = dt.datetime(2026, 3, 16, 0, 0, 0)  # One day before cutover
final_time_4   = dt.datetime(2026, 3, 18, 0, 0, 0)  # One day after cutover
event_df_both = connect_to_db(clean_sql, start_time=initial_time_4, end_time=final_time_4)
event_df_both

/tmp/ipykernel_8738/2540223965.py:4: UserWarning: 
[DATABASE WARNING] The requested time range (2026-03-16 00:00:00 → 2026-03-18 00:00:00) spans the seiscomp3-seiscomp6 database cutover (2026-03-17 00:00:00 UTC). Both databases will be queried and results merged.

  event_df_both = connect_to_db(clean_sql, start_time=initial_time_4, end_time=final_time_4)


,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2026-03-16 00:05:40,SGC2026ffvyal,5.000000,1.124101,0.010000,NaN,NaN,NaN,5.0,5.0,...,NaN,not locatable,SGC,"MutatÃ¡ - Antioquia, Colombia",7.199500,-76.386167,MLr_1,Hypo71,RSNC,None
1,2026-03-16 00:12:09,SGC2026ffwdqf,55.820312,1.307294,0.483821,5.940549,3.685058,5.293564,14.0,14.0,...,NaN,earthquake,SGC,"Hispania - Antioquia, Colombia",5.768267,-75.933496,MLr_1,NonLinLoc,Poveda_et_al_2018,None
2,2026-03-16 00:12:21,SGC2026ffwdtx,29.000000,2.412485,0.900000,3.000000,1.343503,1.343503,44.0,44.0,...,NaN,earthquake,SGC,OcÃ©ano PacÃ­fico,6.161000,-77.876333,MLr_1,Hypo71,RSNC,None
3,2026-03-16 00:37:31,SGC2026ffwzlw,10.000000,0.046732,8.279938,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
4,2026-03-16 00:46:40,SGC2026ffxhio,29.000000,4.181091,0.959388,0.000000,3.982650,4.372885,25.0,23.0,...,13.0,earthquake,SGC,OcÃ©ano PacÃ­fico,4.034341,-82.546722,mb,LOCSAT,iasp91,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458,2026-03-17 22:17:03,SGC2026fjjuen,10.000000,-0.025865,4.241400,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
459,2026-03-17 22:23:34,SGC2026fjjzur,80.000000,1.558878,0.190000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Convención - Norte de Santander, Colombia",8.781667,-73.233000,MLr_3,Hypo71,RSNC,None
460,2026-03-17 22:42:15,SGC2026fjkpxl,5.000000,0.341099,0.100000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Puerto Gaitán - Meta, Colombia",3.882167,-71.381000,MLr_PtoGtn,Hypo71,RSNC,None
461,2026-03-17 23:01:23,SGC2026fjlgjx,5.000000,1.129640,0.060000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Tibasosa - Boyacá, Colombia",5.748000,-73.009833,MLr_3,Hypo71,RSNC,None


In [15]:
# Make a query from 2023 to current time
initial_time = dt.datetime(2023, 1, 1, 0, 0, 0)
final_time = dt.datetime.now(dt.UTC)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)
event_df3

/tmp/ipykernel_8738/852011271.py:4: UserWarning: 
[DATABASE WARNING] The requested time range (2023-01-01 00:00:00 → 2026-07-13 05:25:44) spans the seiscomp3-seiscomp6 database cutover (2026-03-17 00:00:00 UTC). Both databases will be queried and results merged.

  event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)


,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2023-01-01 00:04:23,SGC2023aaaduv,9.355469,1.902402,0.951364,3.270166,1.939846,1.667912,58.0,56.0,...,NaN,earthquake,SGC,"Mesetas - Meta, Colombia",3.460591,-74.178675,MLr_3,NonLinLoc,Poveda_et_al_2018,None
1,2023-01-01 00:21:02,SGC2023aaasdg,10.000000,NaN,9.676469,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Uribe - Meta, Colombia",3.247200,-74.378800,None,,,None
2,2023-01-01 00:31:36,SGC2023aabbgi,0.000000,1.669410,6.302872,10.100000,9.192388,9.192388,7.0,7.0,...,NaN,not locatable,SGC,Mar Caribe,11.858167,-73.487333,M_MLr,Hypo71,RSNC,None
3,2023-01-01 00:45:28,SGC2023aabnex,22.773438,2.467139,1.013426,2.863049,2.649246,3.378211,66.0,56.0,...,NaN,earthquake,SGC,"NunchÃ­a - Casanare, Colombia",5.457120,-72.123158,MLr_3,NonLinLoc,Poveda_et_al_2018,None
4,2023-01-01 00:53:41,SGC2023aabugt,10.000000,NaN,2.477445,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,Mar Caribe,13.376200,-81.363500,None,,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254620,2026-07-13 04:41:42,SGC2026nqqmpu,10.000000,0.032656,14.546030,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"La Esperanza - Norte de Santander, Colombia",7.705400,-73.334400,MLr_vmm,,,None
254621,2026-07-13 04:45:06,SGC2026nqqpoi,5.000000,0.520666,0.100000,NaN,NaN,NaN,4.0,4.0,...,NaN,not locatable,SGC,"Samaná - Caldas, Colombia",5.686500,-74.922667,MLr_2,Hypo71,RSNC,None
254622,2026-07-13 04:49:14,SGC2026nqqtcp,10.000000,0.048993,5.772779,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
254623,2026-07-13 04:56:20,SGC2026nqqzfq,10.000000,-1.050448,15.260187,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"La Esperanza - Norte de Santander, Colombia",7.705400,-73.334400,MLr_vmm,,,None


# Optimizing Duplicates Search

Actually, the revision routine includes a function to check for duplicates, which is based on the time and geographical distance between events. However, this function is not optimized, since it loops over all the events and compares them one by one. This can be very time-consuming, especially if there are many events in the database. The goal of this section is to optimize the duplicates search, by using a more efficient algorithm that can reduce the time complexity of the search. Let's start by defining the previous function to check duplicates:

In [16]:
columns = ['time_value', 'publicID', 'text', 'depth_value', 'magnitude_value', 'magnitude_type',
           'quality_standardError', 'depth_uncertainty', 'latitude_uncertainty', 'longitude_uncertainty',
           'quality_associatedPhaseCount', 'creationInfo_author', 'event_type', 'creationInfo_agencyID']

def check_duplicates(
        events: pd.DataFrame
):
    """
    Identifies duplicate events based on time and geographical distance.

    Parameters:
    -----------
    events: pandas dataframe
        Seismic data for the given time range. Generally obtained from the connect2mysql function.

    Returns:
    --------
    duplicates: A pandas dataframe
        Table with information about duplicate events.
    """
    # Filter events with the following types
    checks = ["earthquake", "volcanic eruption", "explosion", "outside of network interest"]
    selections = events[events['event_type'].isin(checks)].reset_index(drop=True)

    # Make a pandas dataframe to store the info of the duplicates
    duplicates = pd.DataFrame()

    # Loop over al earthquakes, ordered by time
    for i in trange(len(selections) - 1, desc="Checking duplicates", unit=" events", leave=False):
        event1 = selections.iloc[i]  # Take the i-esim event on the list
        event2 = selections.iloc[i + 1]  # Compared to the next event

        # Check if the events are within 4 seconds of each other
        time_diff = abs((event2['time_value'] - event1['time_value']).total_seconds())
        if time_diff <= 4:
            # Estimate the distance between the two events using haversine formula
            distance = hs.haversine((event1['latitude_value'], event1['longitude_value']),
                                     (event2['latitude_value'], event2['longitude_value']))
            if distance <= 100:  # If both events are within 100 km and 4 seconds, they can be duplicates
                dup_1 = event1[columns].copy()
                dup_1['Observations'] = f'Possible duplicate event of {event2["publicID"]}'
                dup_2 = event2[columns].copy()
                dup_2['Observations'] = f'Possible duplicate event of {event1["publicID"]}'
                # Add the two events to the duplicates dataframe
                duplicates = pd.concat([duplicates, dup_1.to_frame().T], ignore_index=True)
                duplicates = pd.concat([duplicates, dup_2.to_frame().T], ignore_index=True)
    return duplicates

As you can see from the function, there are two criteria to consider an event as a duplicate: the time difference between the two events must be less than or equal to 4 seconds, and the geographical distance between the two events must be less than or equal to 100 km. The function loops over all the events, and compares each event with the next one in the list. If both criteria are met, the two events are considered duplicates, and their information is stored in a new dataframe called duplicates.

The idea of checking only the next event in the list is based on the fact that usually the time difference between earthquakes is greater than 4 seconds, so it is unlikely that two events that are far apart in the list are duplicates. Let's test the results for all the 104030 earthquakes from 2023-01-01 to 2026-06-11, and see how many duplicates we can find and the time it takes to run the function.

In [17]:
time1 = time.time()
duplicates_previous = check_duplicates(event_df3)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_previous)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 112
Time taken to run the function: 19.596545696258545 seconds


## Micro-optimization of the adjacent-only logic

The previous function is based on the adjacent-only logic, which means that it only compares each event with the next one in the list. This is a good approach to reduce the time complexity of the search, but it can be further optimized by using some micro-optimizations. For example, we can pre-extract the relevant columns as NumPy arrays to avoid the overhead of pandas indexing in the loop. We can also accumulate the duplicate rows in a list and create the DataFrame at the end, instead of concatenating in each iteration. Let's implement this micro-optimized version of the adjacent-only logic:

In [18]:
def check_duplicates_adjacent(events: pd.DataFrame) -> pd.DataFrame:
    """
    Identifies duplicate events based on time and geographical distance.
    Same adjacent-only logic as the original, but micro-optimized:
      - Pre-extracted NumPy arrays (avoids per-row iloc overhead)
      - Row accumulation in a list (avoids repeated pd.concat in loop)
      - Single pd.concat at the end

    Time complexity: O(n) comparisons — only i vs i+1
    Parameters:
    -----------
    events : pd.DataFrame
        Seismic data for the given time range.
    Returns:
    --------
    duplicates : pd.DataFrame
        Table with information about duplicate events.
    """
    checks = ["earthquake", "volcanic eruption", "explosion", "outside of network interest"]
    selections = events[events['event_type'].isin(checks)].reset_index(drop=True)

    # Edge case: empty or single-event dataset
    if len(selections) < 2:
        return pd.DataFrame(columns=columns + ['Observations'])

    # --- Pre-extract arrays for fast access ---
    times  = selections['time_value'].values          # numpy datetime64
    lats   = selections['latitude_value'].values.astype(np.float64)
    lons   = selections['longitude_value'].values.astype(np.float64)
    ids    = selections['publicID'].values

    TIME_WINDOW    = np.timedelta64(4, 's')
    DIST_THRESHOLD = 100  # km

    dup_rows = []  # Accumulate rows here, build DF once at the end

    for i in trange(len(selections) - 1, desc="Checking duplicates", unit=" events", leave=False):
        # Time check using numpy (faster than .total_seconds() on Timedelta)
        if abs(times[i + 1] - times[i]) <= TIME_WINDOW:
            distance = hs.haversine((lats[i], lons[i]), (lats[i + 1], lons[i + 1]))
            if distance <= DIST_THRESHOLD:
                row_i = selections.iloc[i][columns].copy()
                row_i['Observations'] = f'Possible duplicate event of {ids[i + 1]}'
                row_j = selections.iloc[i + 1][columns].copy()
                row_j['Observations'] = f'Possible duplicate event of {ids[i]}'
                dup_rows.extend([row_i, row_j])

    # Concat duplicates to one single Dataframe
    if dup_rows:
        duplicates = pd.DataFrame(dup_rows, columns=columns + ['Observations']).reset_index(drop=True)
    else:
        duplicates = pd.DataFrame(columns=columns + ['Observations'])

    return duplicates

In [19]:
# Repeat the test with the adjacent-only logic, but micro-optimized
time1 = time.time()
duplicates_adjacent = check_duplicates_adjacent(event_df3)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_adjacent)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 112
Time taken to run the function: 1.1301758289337158 seconds


In [20]:
# Check if the results are the same
print(f"Number of duplicates found with original adjacent-only: {len(duplicates_previous)}")
print(f"Number of duplicates found with micro-optimized adjacent-only: {len(duplicates_adjacent)}")

Number of duplicates found with original adjacent-only: 112
Number of duplicates found with micro-optimized adjacent-only: 112


In [21]:
# Find differences between the two results
differences = pd.concat([duplicates_previous, duplicates_adjacent, duplicates_adjacent]).drop_duplicates(keep=False)
print(f"Number of differences between original and micro-optimized adjacent-only: {len(differences)}")
differences

Number of differences between original and micro-optimized adjacent-only: 0


,time_value,publicID,text,depth_value,magnitude_value,magnitude_type,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,event_type,creationInfo_agencyID,Observations


As you can see, the results are the same, but the time taken to run the function is significantly reduced. This is because we have avoided the overhead of pandas indexing in the loop, and we have also avoided the repeated concatenation of dataframes in each iteration. The time complexity of this function is still O(n), since we are only comparing each event with the next one in the list, but the constant factors have been reduced, which can make a big difference when there are many events in the dataset.

**There are a significant improvement of 95% in the time taken to run the function.** This micro-optimized version of the adjacent-only logic can be used as a baseline for further optimizations, such as using a more efficient algorithm that can reduce the time complexity of the search even further.

Further improving: We can just calculate all the temporal and spatial distances between all the events in one-single shot using vectorized NumPy operations, and then filter the results to find the duplicates. This will reduce the time complexity of the search to O(n^2), but it will be much faster than the previous versions, since we are using vectorized operations instead of loops. Let's implement this version:

In [22]:
def haversine_np(lat1, lon1, lat2, lon2):
    """
    Fully vectorized haversine distance (km) between paired lat/lon arrays.
    Replaces the scalar `hs.haversine()` calls inside the original loop.
    """
    R = 6371.0088
    lat1r, lat2r = np.radians(lat1), np.radians(lat2)
    dlat = lat2r - lat1r
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def check_duplicates_vectorized(
        events: pd.DataFrame,
        time_window: int,
        dist_threshold: float,
) -> pd.DataFrame:
    """
    Identifies duplicate events based on time proximity and geographical
    distance between temporally-adjacent events (after sorting by time).

    Time complexity: O(n) — all comparisons are vectorized NumPy array
    operations; only the (typically rare) confirmed duplicate pairs are
    iterated in Python to build per-row observation text.

    Parameters
    ----------
    events : pd.DataFrame
        Seismic data for the given time range. Must contain 'time_value',
        'latitude_value', 'longitude_value', and 'publicID' columns.
    time_window : int
        Time window in seconds to detect duplicated events.
    dist_threshold : float
        Distance threshold in kilometers to detect duplicates.

    Returns
    -------
    pd.DataFrame
        Subset of `events` (all original columns preserved) with an added
        'Observations' column, containing only the flagged duplicate rows.
    """
    if len(events) < 2:
        empty = events.copy()
        empty['Observations'] = pd.Series(dtype='object')
        return empty.iloc[0:0]

    sorted_events = events.sort_values('time_value').reset_index(drop=True)

    times = sorted_events['time_value'].to_numpy()
    lats  = sorted_events['latitude_value'].to_numpy(dtype=np.float64)
    lons  = sorted_events['longitude_value'].to_numpy(dtype=np.float64)
    ids   = sorted_events['publicID'].to_numpy()

    time_diff   = np.abs(times[1:] - times[:-1])
    within_time = time_diff <= np.timedelta64(time_window, 's')

    distances   = haversine_np(lats[:-1], lons[:-1], lats[1:], lons[1:])
    within_dist = distances <= dist_threshold

    adjacent_dup = within_time & within_dist
    dup_idx = np.where(adjacent_dup)[0]

    if dup_idx.size == 0:
        empty = events.copy()
        empty['Observations'] = pd.Series(dtype='object')
        return empty.iloc[0:0]

    obs_map: dict[int, list[str]] = {}
    for i in dup_idx:
        obs_map.setdefault(int(i), []).append(f'Possible duplicate event of {ids[i + 1]}')
        obs_map.setdefault(int(i) + 1, []).append(f'Possible duplicate event of {ids[i]}')

    flagged_positions = sorted(obs_map.keys())
    flagged = sorted_events.iloc[flagged_positions].copy()
    flagged['Observations'] = ['; '.join(obs_map[p]) for p in flagged_positions]

    return flagged.reset_index(drop=True)

In [23]:
# Repeat the test with the adjacent-only logic, but vectorized
time1 = time.time()
# Filter events by label:
filtered = event_df3[event_df3['event_type'].isin(["earthquake", "volcanic eruption", "explosion", "outside of network interest"])].reset_index(drop=True)
duplicates_vectorized = check_duplicates_vectorized(filtered, time_window=4, dist_threshold=100)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_vectorized)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 109
Time taken to run the function: 0.1892528533935547 seconds


The apparent discrepancy between the results is due to the fact that the vectorized version joins the observations for each event into a single string, while the previous versions created separate rows for each observation. This means that if an event is flagged as a duplicate of multiple other events, it will appear only once in the vectorized result, with all relevant observations concatenated. In contrast, the previous versions would create multiple rows for that event, one for each duplicate relationship.

Ignoring this difference in representation, the underlying logic for identifying duplicates is consistent across all implementations. The vectorized approach is significantly faster due to its use of NumPy operations, which are optimized for performance and can handle large datasets more efficiently than iterative loops in Python.

## Sorted Sliding Window Approach

Now, let's use a improved version created using the Sorted Sliding Window approach. We will also compare the results with the previous version of the function, which was based on a nested loop that compared all events with each other, and see how much time we can save with the new algorithm.

In [24]:
def check_duplicates_2(events: pd.DataFrame) -> pd.DataFrame:
    """
        Identifies duplicate events based on time and geographical distance. Optimized using a sorted sliding time window (O(n log n)).

    Parameters:
    -----------
    events : pd.DataFrame
        Seismic data for the given time range.

    Returns:
    --------
    duplicates : pd.DataFrame
        Table with information about duplicate events.
    """
    checks = ["earthquake", "volcanic eruption", "explosion", "outside of network interest"]
    selections = (
        events[events['event_type'].isin(checks)]
        .sort_values('time_value')
        .reset_index(drop=True)
    )

    # Edge case: empty or single-event dataset
    if len(selections) < 2:
        return pd.DataFrame(columns=columns + ['Observations'])

    # Pre-extract arrays for fast access (avoids per-row iloc overhead)
    times = selections['time_value'].values
    lats  = selections['latitude_value'].values
    lons  = selections['longitude_value'].values
    ids   = selections['publicID'].values

    TIME_WINDOW   = 4    # seconds
    DIST_THRESHOLD = 100  # km

    # Track which (i, j) pairs were already flagged to avoid duplicate rows
    flagged_pairs = set()
    dup_rows = []

    left = 0  # Left pointer of the sliding window

    for right in trange(1, len(selections), desc="Checking duplicates", unit=" events", leave=False):
        # Advance left pointer: drop events outside the 4-second window
        while left < right:
            delta = (times[right] - times[left]) / pd.Timedelta('1s')
            if delta > TIME_WINDOW:
                left += 1
            else:
                break

        # Compare `right` against every event still inside the window [left, right-1]
        for i in range(left, right):
            pair = (i, right) if i < right else (right, i)
            if pair in flagged_pairs:
                continue

            time_diff = abs(
                (selections.iloc[right]['time_value'] - selections.iloc[i]['time_value'])
                .total_seconds()
            )
            if time_diff > TIME_WINDOW:
                continue

            distance = hs.haversine((lats[i], lons[i]), (lats[right], lons[right]))
            if distance <= DIST_THRESHOLD:
                flagged_pairs.add(pair)

                row_i = selections.iloc[i][columns].copy()
                row_i['Observations'] = f'Possible duplicate event of {ids[right]}'

                row_j = selections.iloc[right][columns].copy()
                row_j['Observations'] = f'Possible duplicate event of {ids[i]}'

                dup_rows.extend([row_i, row_j])

    # Build duplicates dataframe in one shot (avoids repeated pd.concat overhead)
    if dup_rows:
        duplicates = pd.DataFrame(dup_rows, columns=columns + ['Observations']).reset_index(drop=True)
    else:
        duplicates = pd.DataFrame(columns=columns + ['Observations'])

    return duplicates

In [25]:
time1 = time.time()
duplicates_sstw = check_duplicates_2(event_df3)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_sstw)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 112
Time taken to run the function: 2.485229015350342 seconds


As you can see from here, the time taken to run the function is slightly higher than the micro-optimized adjacent-only version (51% slower), but it is still significantly lower than the original nested loop version (90% faster). Moreover, this method can find duplicates that are not adjacent in the list, as long as they are within the 4-second time window. This can be useful in cases where there are multiple events that occur within a short time frame, but are not necessarily adjacent in the list.

The time complexity of this function is O(n log n) due to the initial sorting step, and O(n * k) for the sliding window comparisons, where k is the average number of events within the 4-second window. In practice, this can be much faster than the O(n^2) complexity of the original nested loop approach, especially when there are many events in the dataset. Let's check if the duplicates found with this new approach are the same as the ones found with the adjacent-only logic, and see if there are any additional duplicates that were not found with the previous method.

In [26]:
# Check if the results are the same
print(f"Number of duplicates found with adjacent-only: {len(duplicates_adjacent)}")
print(f"Number of duplicates found with sorted sliding window: {len(duplicates_sstw)}")
differences = pd.concat([duplicates_sstw, duplicates_adjacent, duplicates_adjacent]).drop_duplicates(keep=False)
print(f"Number of additional duplicates found with sorted sliding window: {len(differences)}")
differences

Number of duplicates found with adjacent-only: 112
Number of duplicates found with sorted sliding window: 112
Number of additional duplicates found with sorted sliding window: 0


,time_value,publicID,text,depth_value,magnitude_value,magnitude_type,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,event_type,creationInfo_agencyID,Observations


Now, let's implement a fully vectorized version of the sorted sliding window approach. This will allow us to leverage NumPy's efficient array operations to further reduce the time complexity of the search. The idea is to precompute the time and distance matrices for all events, and then filter the results based on the specified time and distance thresholds. This will allow us to identify duplicate events without the need for nested loops, resulting in a significant speedup for large datasets.

In [34]:
def check_duplicates_2_vectorized(
        events: pd.DataFrame,
        columns: list[str],
        time_window: int,
        dist_threshold: float,
) -> pd.DataFrame:
    """
    Identifies duplicate events based on time proximity and geographical
    distance, using a sliding time window (compares every event against
    ALL others within time_window seconds, not just its direct neighbor).

    Parameters
    ----------
    events : pd.DataFrame
        Seismic data for the given time range. Must contain the columns
        referenced by `columns`.
    columns : list[str]
        Ordered list [time_col, lat_col, lon_col, id_col] identifying which
        columns hold time, latitude, longitude, and event ID values.
    time_window : int
        Time window in seconds to detect duplicated events.
    dist_threshold : float
        Distance threshold in kilometers to detect duplicates.

    Returns
    -------
    pd.DataFrame
        Subset of `events` (all original columns preserved) with an added
        'Observations' column, one row per flagged duplicate event —
        chains of 3+ nearby events are merged into a single row per event
        rather than duplicated across multiple pair-rows.
    """
    time_col, lat_col, lon_col, id_col = columns
    selections = events.copy()

    if len(selections) < 2:
        empty = events.copy()
        empty['Observations'] = pd.Series(dtype='object')
        return empty.iloc[0:0]

    sorted_events = selections.sort_values(time_col).reset_index(drop=True)

    times = sorted_events[time_col].to_numpy()
    lats  = sorted_events[lat_col].to_numpy(dtype=np.float64)
    lons  = sorted_events[lon_col].to_numpy(dtype=np.float64)
    ids   = sorted_events[id_col].to_numpy()
    n     = len(sorted_events)

    window_delta = np.timedelta64(time_window, 's')

    # For every i, right_bound[i] = first index j where times[j] > times[i] + window
    right_bound = np.searchsorted(times, times + window_delta, side='right')

    obs_map: dict[int, list[str]] = {}

    for i in range(n - 1):
        j_end = right_bound[i]
        if j_end <= i + 1:
            continue

        window_lats = lats[i + 1:j_end]
        window_lons = lons[i + 1:j_end]
        distances = haversine_np(lats[i], lons[i], window_lats, window_lons)

        matched_offsets = np.where(distances <= dist_threshold)[0]
        if matched_offsets.size == 0:
            continue

        matched_js = i + 1 + matched_offsets
        for j in matched_js:
            obs_map.setdefault(i, []).append(f'Possible duplicate event of {ids[j]}')
            obs_map.setdefault(int(j), []).append(f'Possible duplicate event of {ids[i]}')

    if not obs_map:
        empty = events.copy()
        empty['Observations'] = pd.Series(dtype='object')
        return empty.iloc[0:0]

    flagged_positions = sorted(obs_map.keys())
    flagged = sorted_events.iloc[flagged_positions].copy()
    flagged['Observations'] = ['; '.join(obs_map[p]) for p in flagged_positions]

    return flagged.reset_index(drop=True)

In [35]:
# Repeat the test with the vectorized sswa
time1 = time.time()
duplicates_vectorized_sswa = check_duplicates_2_vectorized(filtered, time_window=4, dist_threshold=100, columns=['time_value', 'latitude_value', 'longitude_value', 'publicID'])
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_vectorized_sswa)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 109
Time taken to run the function: 0.13587474822998047 seconds


### Overall results

Summarizing the results of the different duplicate detection methods, we have for 106700 events:

1. Original, for loop version: 21.64 s
2. Original, micro-optimized adjacent-only: 1.05 s
3. Vectorized adjacent-only: 0.16 s
4. Sorted Sliding Window: 2.60 s
5. Vectorized Sorted Sliding Window: 0.13 s

Therefore, the overall recommendation is to use the vectorized adjacent-only version if the time window is similar to the mean temporal separation between events, since it is the fastest and most efficient method. However, if the time window is larger than the mean temporal separation between events, the sorted sliding window approach may be more appropriate, as it can detect duplicates that are not adjacent in the list.